In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


- Split data into train and test
- Write helper functions
    - One for training and scaling
    - One for CV, tuning, eval
- Create and train models
- Comparison

In [27]:
df = pd.read_csv('cleaned_apartment_data.csv')


In [28]:
df.columns


Index(['title', 'body', 'amenities', 'bathrooms', 'bedrooms', 'fee',
       'pets_allowed', 'price', 'square_feet', 'cityname', 'latitude',
       'longitude', 'time', 'state_AK', 'state_AL', 'state_AR', 'state_AZ',
       'state_CA', 'state_CO', 'state_CT', 'state_DC', 'state_DE', 'state_FL',
       'state_GA', 'state_HI', 'state_IA', 'state_ID', 'state_IL', 'state_IN',
       'state_KS', 'state_KY', 'state_LA', 'state_MA', 'state_MD', 'state_ME',
       'state_MI', 'state_MN', 'state_MO', 'state_MS', 'state_MT', 'state_NC',
       'state_ND', 'state_NE', 'state_NH', 'state_NJ', 'state_NM', 'state_NV',
       'state_NY', 'state_OH', 'state_OK', 'state_OR', 'state_PA', 'state_RI',
       'state_SC', 'state_SD', 'state_TN', 'state_TX', 'state_UT', 'state_VA',
       'state_VT', 'state_WA', 'state_WI', 'state_WV', 'state_WY', 'state',
       'date', 'month', 'pool', 'dishwasher', 'gym', 'parking', 'fireplace',
       'patio/deck', 'storage', 'has_photo_thumbnail_only', 'has_photo'],
    

In [47]:
df['price'].describe()

count    99331.000000
mean      1525.241475
std        883.418048
min        100.000000
25%       1014.000000
50%       1350.000000
75%       1795.000000
max      40000.000000
Name: price, dtype: float64

In [30]:
# list of variables not to include
non_vars = [
    'title',
    'body',
    'amenities',
    'cityname',
    'time',
    'state',
    'state_TX', # Use as baseline to avoid multicolinearity, Texas most common state
    'date',
    'month',
    'price' # Target y variable
]

X = df.drop(columns=non_vars)
y = df['price']

In [46]:
len(X.columns)

66

In [31]:
# Set random seed to 42

seed = 42

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=seed
)

In [ ]:
# First helper function, create pipeline with base model
# Pipeline allows scaling without leakage, and standardizes syntax in function 2

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def base_pipeline(model, scale=False):
    steps = []

    steps.append(('model', model))

    if scale:
        steps.append(('scaler', StandardScaler()))

    return Pipeline(steps)
    

In [39]:
# Second helper function, use CV to fine tune, and output eval metrics
# EACH TIME YOU CREATE PARAM_GRID MUST USE 'model__' SYNTAX IN DICT KEYS

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_squared_error

def tune_eval(pipeline, param_grid, X_train, X_test, y_train, y_test):
    # CV
    model_tuned = RandomizedSearchCV(
        estimator = pipeline, 
        param_distributions = param_grid,
        cv=5,
        n_iter=15,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=seed
    )
    model_tuned.fit(X_train, y_train)

    # predictions
    y_pred = model_tuned.predict(X_test)

    # Test metrics
    test_mse = mean_squared_error(y_test, y_pred)
    test_rmse = np.sqrt(test_mse)

    # output eval metrics
    print('Best model parameters:', model_tuned.best_params_)
    print('Best CV MSE:', np.abs(model_tuned.best_score_))
    print('Best CV RMSE:', np.sqrt(np.abs(model_tuned.best_score_)))
    print('Test MSE:', test_mse)
    print('Test RMSE:', test_rmse)

    # Return the final tuned model
    return model_tuned.best_estimator_

## Decision Tree Regression

- Look at either dropping outliers or log-transforming price???

In [43]:
# Instantiate model, default vales, and param_grid for DT

from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=seed)

dt_param_grid = {
    'model__max_depth': np.linspace(2, 20, num=10).astype(int),
    'model__min_samples_leaf': np.linspace(5, 200, num=10).astype(int) 
}

In [44]:
# Helper function 1

dt_pipeline = base_pipeline(dt)

# Helper function 2

tune_eval(dt_pipeline, dt_param_grid, X_train, X_test, y_train, y_test)

Best model parameters: {'model__min_samples_leaf': np.int64(5), 'model__max_depth': np.int64(20)}
Best CV MSE: 245931.18788536047
Best CV RMSE: 495.9144965468951
Test MSE: 219274.05271226302
Test RMSE: 468.267074127856


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",np.int64(20)
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a f